# Phase C - ML (case/control on expression + communication features)

**Scaffold for Andreas.** Donor-level classification of MDD vs Control using
pseudobulk expression and the Phase B communication factors, with **rigorous,
leakage-safe CV**: the tensor is refit on *train donors only* each fold and test
donors are projected onto the fixed factors. Evaluation = within-sex value-add
(see the Phase B handoff). Cells marked `TODO(Andreas)` are choices left to you.

### Dependencies

In [ ]:
# liana + cell2cell are needed for the in-fold tensor refit (rigorous path).
!pip install -q scanpy pyarrow liana cell2cell xgboost shap scikit-learn

### Boot cell (mount Drive, pull repo)

In [ ]:
import os, sys
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT   = '/content/drive/MyDrive/MLCB_team_project'
CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, 'data', 'checkpoints')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

REPO_URL = 'https://github.com/andreas824/MLCB_team_project.git'
REPO_DIR = '/content/MLCB_team_project'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
# autoreload is optional (Colab 3.12 + IPython 7.34 ship a broken one)
try:
    _ip = get_ipython()
    _ip.run_line_magic('load_ext', 'autoreload')
    _ip.run_line_magic('autoreload', '2')
except Exception as e:
    print(f'autoreload unavailable ({e}); continuing.')

### Step 0 - load metadata, labels, communication inputs

In [ ]:
# === Phase C - Step 0: metadata, labels, communication inputs ===
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd

CHECKPOINT_DIR = globals().get('CHECKPOINT_DIR',
    '/content/drive/MyDrive/MLCB_team_project/data/checkpoints')

# donor metadata + stratification label (sex x diagnosis)
obs = pd.read_parquet(os.path.join(CHECKPOINT_DIR, 'phaseA_obs.parquet'))
meta = obs.drop_duplicates('donor_id').set_index('donor_id')[['condition', 'sex']]
meta['y'] = (meta['condition'] == 'MDD').astype(int)
meta['strat'] = meta['sex'].astype(str) + '_' + meta['condition'].astype(str)
print(meta['strat'].value_counts().to_string())

# full-data communication factors (Phase B) - for interpretation / SHAP only
donor_factors = pd.read_parquet(os.path.join(CHECKPOINT_DIR, 'phaseB_donor_factors.parquet'))
print('precomputed donor factors:', donor_factors.shape)

# per-donor LIANA results - rebuilt into tensors per CV fold (the rigorous path)
liana_res = pd.read_parquet(os.path.join(CHECKPOINT_DIR, 'phaseB_liana_per_donor.parquet'))
print('liana_res:', liana_res.shape)

### Step 1 - pseudobulk expression per donor

Sum raw counts per donor (from the Phase A `.layers['counts']`), then CPM +
log1p. Both are per-donor operations, so doing them globally is leakage-safe;
**HVG selection and scaling happen in-fold** (Step 3). Needs the 5.8 GB h5ad ->
**CPU + High-RAM**. Cached so you load it once.

In [ ]:
# === Phase C - Step 1: pseudobulk expression per donor (raw -> logCPM) ===
import glob
import scipy.sparse as sp

pb_cache = os.path.join(CHECKPOINT_DIR, 'phaseC_pseudobulk_logcpm.parquet')
if os.path.exists(pb_cache):
    expr = pd.read_parquet(pb_cache)
    print('loaded cached pseudobulk:', expr.shape)
else:
    import scanpy as sc
    pats = [os.path.join(CHECKPOINT_DIR, 'phaseA_final_for_cellchat.h5ad'),
            '/content/drive/MyDrive/**/phaseA*cellchat*.h5ad']
    CKPT = next((g for p in pats for g in sorted(glob.glob(p, recursive=True))), None)
    adata = sc.read_h5ad(CKPT)
    dcat = adata.obs['donor_id'].astype('category')
    codes = dcat.cat.codes.values
    Dmat = sp.csr_matrix((np.ones(adata.n_obs), (codes, np.arange(adata.n_obs))),
                         shape=(dcat.cat.categories.size, adata.n_obs))
    pb = np.asarray(Dmat @ adata.layers['counts'])         # (donor, gene) raw summed
    lib = pb.sum(1, keepdims=True)
    expr = pd.DataFrame(np.log1p(pb / lib * 1e6),
                        index=dcat.cat.categories, columns=adata.var_names.astype(str))
    expr.to_parquet(pb_cache)
    del adata
    print('pseudobulk expression:', expr.shape, '-> cached')

### Step 2 - rigorous in-fold communication features

Build the full 4D tensor once. Each CV fold refits the CP factors on **train
donors only** and **projects** test donors onto the fixed (LR/sender/receiver)
factors via non-negative least squares - so test donors never shape the factor
space. The self-check at the end (projecting train donors ~ their fitted
loadings) validates the projection math.

In [ ]:
# === Phase C - Step 2: rigorous in-fold communication features ===
import liana as li
from cell2cell.tensor.tensor import PreBuiltTensor
from scipy.optimize import nnls

RANK = 5
ORDER_LABELS = ['Contexts', 'Ligand-Receptor Pairs', 'Sender Cells', 'Receiver Cells']

ft = li.multi.to_tensor_c2c(liana_res=liana_res, sample_key='donor_id',
                            score_key='magnitude_rank', how='outer_cells')   # build (minutes)
T = np.asarray(ft.tensor, dtype=float)        # (donor, LR, sender, receiver); NaN where masked
donor_order = list(ft.order_names[0])
pos_of = {d: i for i, d in enumerate(donor_order)}
print('full tensor:', T.shape, '| donors:', len(donor_order))

def fit_factors(train_pos, rank=RANK, seed=0):
    # CP/NNTF on the TRAIN sub-tensor (mask preserved)
    sub = T[train_pos]
    names = [[donor_order[i] for i in train_pos]] + ft.order_names[1:]
    mask = (~np.isnan(sub)).astype(int)
    t = PreBuiltTensor(tensor=np.nan_to_num(sub), order_names=names,
                       order_labels=ORDER_LABELS, mask=mask)
    t.compute_tensor_factorization(rank=rank, init='random', random_state=seed)
    return t

def _design(fitted):
    # M[:, r] = vec(B[:,r] x C[:,r] x D[:,r]) in the SAME (LR,send,recv) C-order
    # as T[i].reshape(-1) -> guarantees row alignment for the NNLS fit.
    B = fitted.factors['Ligand-Receptor Pairs'].values
    C = fitted.factors['Sender Cells'].values
    D = fitted.factors['Receiver Cells'].values
    M = np.empty((B.shape[0] * C.shape[0] * D.shape[0], B.shape[1]))
    for r in range(B.shape[1]):
        M[:, r] = np.einsum('i,j,k->ijk', B[:, r], C[:, r], D[:, r]).reshape(-1)
    return M

def project(test_pos, fitted):
    # NNLS projection of each test donor onto the fixed factors (masked entries dropped)
    M = _design(fitted)
    out = np.zeros((len(test_pos), fitted.factors['Contexts'].shape[1]))
    for k, pos in enumerate(test_pos):
        s = T[pos].reshape(-1)
        m = np.isfinite(s)
        out[k], _ = nnls(M[m], s[m])
    return out

# self-check: projecting train donors should ~reproduce their fitted loadings
_f = fit_factors(list(range(len(donor_order))), rank=RANK)
_p = project(list(range(len(donor_order))), _f)
print('projection self-check corr:',
      round(float(np.corrcoef(_p.ravel(), _f.factors['Contexts'].values.ravel())[0, 1]), 3))

### Step 3 - rigorous donor-level CV (within-sex-stratified)

Compare **expression-only vs communication-only vs combined**. Per fold: comm
features refit on train + project test (no leakage); expression HVG + scaling
fit on train only. Pooled CV stratified by sex x diagnosis (diagnosis is
balanced within sex, so the split never crosses the cohort/batch axis).

**Heavy:** refits the tensor every fold. Start with `n_repeats=2` to smoke-test,
then raise. `TODO(Andreas)`: wrap XGB in nested CV for tuning; report per-sex
AUC; add a label-shuffle null.

In [ ]:
# === Phase C - Step 3: rigorous donor-level CV ===
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

N_HVG = 1000          # TODO(Andreas): tune; fit on train each fold
XGB = dict(n_estimators=200, max_depth=3, learning_rate=0.05,
           subsample=0.8, colsample_bytree=0.8, eval_metric='logloss')

donors = [d for d in donor_order if d in expr.index and d in meta.index]
y = meta.loc[donors, 'y'].values
strat = meta.loc[donors, 'strat'].values
E = expr.loc[donors].values

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=0)   # raise n_repeats later
scores = {k: [] for k in ['expr', 'comm', 'combined']}
for tr, te in cv.split(donors, strat):
    tr_pos = [pos_of[donors[i]] for i in tr]
    te_pos = [pos_of[donors[i]] for i in te]
    # communication: rigorous refit on train, project test
    fitted = fit_factors(tr_pos, rank=RANK, seed=0)
    Ctr, Cte = fitted.factors['Contexts'].values, project(te_pos, fitted)
    sc_c = StandardScaler().fit(Ctr); Ctr, Cte = sc_c.transform(Ctr), sc_c.transform(Cte)
    # expression: HVG + scale fit on train only
    Etr, Ete = E[tr], E[te]
    hvg = np.argsort(Etr.var(0))[::-1][:N_HVG]
    sc_e = StandardScaler().fit(Etr[:, hvg])
    Etr, Ete = sc_e.transform(Etr[:, hvg]), sc_e.transform(Ete[:, hvg])
    sets = {'expr': (Etr, Ete), 'comm': (Ctr, Cte),
            'combined': (np.hstack([Etr, Ctr]), np.hstack([Ete, Cte]))}
    for name, (A, Bm) in sets.items():
        clf = XGBClassifier(**XGB).fit(A, y[tr])
        scores[name].append(roc_auc_score(y[te], clf.predict_proba(Bm)[:, 1]))

print('rigorous donor-level CV (AUC, mean +/- std):')
for name in ['expr', 'comm', 'combined']:
    s = np.array(scores[name]); print(f'  {name:9s} {s.mean():.3f} +/- {s.std():.3f}')
print('\nValue-add = combined - expr.')

### Step 4 - interpretation: SHAP + shared-across-sex importance (stub)

The shared-axis test: is the microglia->ExN10_L46 program (**Factor 4**)
important for case/control in **both** sexes? Uses the full-data factors for
interpretation. NOTE: final-model SHAP on all donors carries the mild
unsupervised-tensor caveat - state it in the report.

In [ ]:
# === Phase C - Step 4: SHAP + within-sex shared importance (stub) ===
import shap   # TODO(Andreas)

# Skeleton:
#   X = combined features on all donors (expression HVG fit on all -> interpretation only)
#   clf = XGBClassifier(**XGB).fit(X, y)
#   sv = shap.TreeExplainer(clf).shap_values(X)
#   - global importance ranking (genes + Factor 1..5)
#   - Factor 4 importance OVERALL and computed WITHIN females and WITHIN males
#     -> shared importance across sexes = evidence for the cross-sex axis
print('TODO(Andreas): final-model SHAP; show Factor 4 importance is shared across sexes.')